# 📅 Exercício Extra 1: O Detetor de Linha Ótico 🏎️🛣️

Neste exercício, vamos preparar o robô para seguir uma linha no chão (ex: fita isoladora amarela ou preta). Em vez de apenas olhar para uma cor, o robô vai calcular a **posição exata (o centro)** dessa linha no ecrã.

Usando a matemática dos **Momentos da Imagem**, o OpenCV consegue encontrar o "centro de massa" da zona branca da máscara.

### 🎯 O Teu Objetivo
Calcular a diferença (o **Erro**) entre o centro da imagem e o centro da linha. Se o erro for positivo, o robô precisa de virar para um lado; se for negativo, para o outro.

### 🛠️ Instruções
1. Garante que as **rodas estão no ar**.
2. Coloca uma fita colorida no chão à frente da câmara.
3. Executa o código e ajusta os sliders para isolar a cor da tua fita.
4. Move a fita para a esquerda e para a direita à frente da câmara. Vais ver uma linha vermelha desenhada no ecrã a marcar o centro detetado e o valor do erro a mudar no texto!

In [ ]:
import cv2
import numpy as np
import ipywidgets as widgets
from IPython.display import display
from jetcam.csi_camera import CSICamera
import time

print("--- DETETOR DE LINHA ÓTICO ATIVO ---")

camera = CSICamera(width=300, height=300, capture_width=1280, capture_height=720, capture_fps=15)
imagem_widget = widgets.Image(format='jpeg', width=300, height=300)
slider_h_min = widgets.IntSlider(value=20, min=0, max=179, description='H Mínimo:')
slider_h_max = widgets.IntSlider(value=40, min=0, max=179, description='H Máximo:')
botao_desligar = widgets.Button(description="❌ DESLIGAR", button_style='danger')

display(imagem_widget, widgets.VBox([slider_h_min, slider_h_max, botao_desligar]))

sistema_ativo = True

def processar_linha(change):
    global sistema_ativo
    if not sistema_ativo: return
    
    frame = change['new']
    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
    baixo = np.array([slider_h_min.value, 100, 100])
    alto = np.array([slider_h_max.value, 255, 255])
    mascara = cv2.inRange(hsv, baixo, alto)
    
    # Calcular os Momentos da imagem filtrada para achar o centro
    M = cv2.moments(mascara)
    
    centro_imagem_x = 150 # Metade da largura (300/2)
    
    if M["m00"] > 0:
        # Fórmula matemática para achar o Centro X da linha
        centro_linha_x = int(M["m10"] / M["m00"])
        
        # --- 🎯 CÁLCULO DO ERRO ---
        erro = centro_linha_x - centro_imagem_x
        print(f"Linha detetada no X: {centro_linha_x} | ERRO: {erro}      ", end='\r')
        
        # Desenha uma linha guia visual no ecrã para os alunos verem
        cv2.line(frame, (centro_linha_x, 0), (centro_linha_x, 300), (0, 0, 255), 3)
    else:
        print("⚠️ Linha não encontrada no campo de visão!          ", end='\r')
        
    _, jpeg = cv2.imencode('.jpg', frame)
    imagem_widget.value = jpeg.tobytes()
    time.sleep(0.02)

camera.observe(processar_linha, names='value')

def encerra(b):
    global sistema_ativo
    sistema_ativo = False
    camera.unobserve(processar_linha, names='value')
    camera.running = False
    print("\nSistema desligado.")

botao_desligar.on_click(encerra)
camera.running = True